In [0]:
# ==========================================
# GOLD TEAM PERFORMANCE - SOURCE INSPECTION
# ==========================================

matches_df = spark.table(
    "workspace.default.silver_matches"
)

teams_df = spark.table(
    "workspace.default.silver_teams"
)

# Check record counts
print("Total matches:", matches_df.count())
print("Total teams:", teams_df.count())

# Inspect match statuses
matches_df.groupBy("match_status").count().show()

# Inspect available columns
matches_df.printSchema()

# Preview completed matches
display(
    matches_df.filter(
        matches_df.match_status == "FINISHED"
    ).limit(10)
)


In [0]:
from pyspark.sql.functions import col, when, lit

# ==========================================
# 1. FILTER COMPLETED MATCHES
# ==========================================

finished_df = matches_df.filter(
    (col("match_status") == "FINISHED") &
    col("home_goals").isNotNull() &
    col("away_goals").isNotNull()
)

print("Finished matches:", finished_df.count())

display(finished_df.limit(10))


In [0]:
from pyspark.sql.functions import sum as spark_sum

# ==========================================
# 2. HOME TEAM PERFORMANCE
# ==========================================

home_df = finished_df.groupBy(
    col("season_id"),
    col("home_team_id").alias("team_id")
).agg(

    # Total home matches
    spark_sum(lit(1)).alias("home_played"),

    # Home wins
    spark_sum(
        when(col("home_goals") > col("away_goals"), 1).otherwise(0)
    ).alias("home_wins"),

    # Home draws
    spark_sum(
        when(col("home_goals") == col("away_goals"), 1).otherwise(0)
    ).alias("home_draws"),

    # Home losses
    spark_sum(
        when(col("home_goals") < col("away_goals"), 1).otherwise(0)
    ).alias("home_losses"),

    # Goals scored and conceded
    spark_sum("home_goals").alias("home_goals_for"),
    spark_sum("away_goals").alias("home_goals_against"),

    # Home clean sheets
    spark_sum(
        when(col("away_goals") == 0, 1).otherwise(0)
    ).alias("home_clean_sheets")
)

display(home_df)


In [0]:
# ==========================================
# 3. AWAY TEAM PERFORMANCE
# ==========================================

away_df = finished_df.groupBy(
    col("season_id"),
    col("away_team_id").alias("team_id")
).agg(

    spark_sum(lit(1)).alias("away_played"),

    spark_sum(
        when(col("away_goals") > col("home_goals"), 1).otherwise(0)
    ).alias("away_wins"),

    spark_sum(
        when(col("away_goals") == col("home_goals"), 1).otherwise(0)
    ).alias("away_draws"),

    spark_sum(
        when(col("away_goals") < col("home_goals"), 1).otherwise(0)
    ).alias("away_losses"),

    spark_sum("away_goals").alias("away_goals_for"),
    spark_sum("home_goals").alias("away_goals_against"),

    spark_sum(
        when(col("home_goals") == 0, 1).otherwise(0)
    ).alias("away_clean_sheets")
)

display(away_df)


In [0]:
from pyspark.sql.functions import round, coalesce

# ==========================================
# 4. JOIN HOME AND AWAY PERFORMANCE
# ==========================================

# Join using team and season identifiers
combined_df = home_df.alias("h").join(
    away_df.alias("a"),
    ["season_id", "team_id"],
    "full"
)

# Replace missing numerical statistics with zero
stat_columns = [
    c for c in combined_df.columns
    if c not in ["season_id", "team_id"]
]

combined_df = combined_df.fillna(
    0, subset=stat_columns
)

# Add team names
combined_df = combined_df.join(
    teams_df.select("team_id", "team_name"),
    "team_id",
    "left"
)

# Calculate points and percentages
gold_df = (
    combined_df
    .withColumn(
        "home_points",
        col("home_wins") * 3 + col("home_draws")
    )
    .withColumn(
        "away_points",
        col("away_wins") * 3 + col("away_draws")
    )
    .withColumn(
        "total_played",
        col("home_played") + col("away_played")
    )
    .withColumn(
        "total_points",
        col("home_points") + col("away_points")
    )
    .withColumn(
        "home_win_pct",
        when(
            col("home_played") > 0,
            round(col("home_wins") / col("home_played") * 100, 2)
        )
    )
    .withColumn(
        "away_win_pct",
        when(
            col("away_played") > 0,
            round(col("away_wins") / col("away_played") * 100, 2)
        )
    )
)

display(gold_df.orderBy("team_name"))


In [0]:
# ==========================================
# 5. GOLD DATA QUALITY CHECKS
# ==========================================

total_matches = finished_df.count()

total_home_matches = gold_df.agg(
    spark_sum("home_played")
).first()[0]

total_away_matches = gold_df.agg(
    spark_sum("away_played")
).first()[0]

# Each completed match has one home and one away team
assert total_home_matches == total_matches, \
    "Home match count does not match source!"

assert total_away_matches == total_matches, \
    "Away match count does not match source!"

# Ensure team-season records are unique
total_rows = gold_df.count()

unique_rows = gold_df.select(
    "season_id", "team_id"
).distinct().count()

assert total_rows == unique_rows, \
    "Duplicate team-season records detected!"

# Ensure no missing team names
assert gold_df.filter(
    col("team_name").isNull()
).count() == 0, "Missing team names!"

# Verify match results add up
invalid_results = gold_df.filter(
    (col("home_played") !=
     col("home_wins") + col("home_draws") + col("home_losses")) |
    (col("away_played") !=
     col("away_wins") + col("away_draws") + col("away_losses"))
).count()

assert invalid_results == 0, \
    "Home/away results are inconsistent!"

print("Gold team performance validation passed!")
print("Finished matches processed:", total_matches)
print("Teams in Gold table:", total_rows)


In [0]:
# ==========================================
# 6. SAVE GOLD TEAM PERFORMANCE
# ==========================================

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_team_performance")
)

print("Gold team performance table created!")


In [0]:
%sql
SELECT
    team_name,
    home_played,
    home_wins,
    home_draws,
    home_losses,
    home_points,
    away_played,
    away_wins,
    away_draws,
    away_losses,
    away_points,
    total_played,
    total_points,
    home_win_pct,
    away_win_pct
FROM workspace.default.gold_team_performance
ORDER BY total_points DESC, team_name;
